# MCP Server

- usually, we either built MPC Server or Clients
    - Servers are for allowing other MCP clients to connect to us to use our tools and functionalities
    - Clients are used to connect to other MCP servers to use their tools and functionalities

In [ ]:
from mcp.server.fastmcp import FastMCP

# Create MCP server (just this one line)
mcp = FastMCP("DocumentMCP", log_level="ERROR")

In [ ]:
docs = {
    "deposition.md": "This deposition covers the testimony of Angela Smith, P.E.",
    "report.pdf": "The report details the state of a 20m condenser tower.",
    "financials.docx": "These financials outline the project's budget and expenditures.",
    "outlook.pdf": "This document presents the projected future performance of the system.",
    "plan.md": "The plan outlines the steps for the project's implementation.",
    "spec.txt": "These specifications define the technical requirements for the equipment.",
}

# Defining Tools

In [ ]:
from pydantic import Field

# @mcp.tool decorator allows us claude to automatically define JSON Schema for tool
@mcp.tool(
    name="read_doc_contents",
    description="Read the contents of a document and return it as a string."
)
def read_document(
    doc_id: str = Field(description="Id of the document to read")
):
    # Error handling to give claude meaningful error to retry and solve error
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")
    # Return the actual doc
    return docs[doc_id]

# second tool with same concept
@mcp.tool(
    name="edit_document",
    description="Edit a document by replacing a string in the documents content with a new string."
)
def edit_document(
    doc_id: str = Field(description="Id of the document that will be edited"),
    old_str: str = Field(description="The text to replace. Must match exactly, including whitespace."),
    new_str: str = Field(description="The new text to insert in place of the old text.")
):
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")
    
    docs[doc_id] = docs[doc_id].replace(old_str, new_str)

# Testing tool
- in bash, write: `mcp dev mcp_server.py`
- you will find port and url, click on url to test server

# Defining Resources

2 functions:

- When users type '@', MCP server will give back a list of files to add to context
- When users mention a file e.g. @4_MCP.ipynb, the MCP server will get the content of that single file

types of resources:

1) Direct - URI doesnt contain params
2) Templated - URI calls are dynamic, based on what the doc_id is

In [ ]:
@mcp.resource(
        "docs://documents", 
        mime_type="application/json"
) # first arg is the URI, second arg is the mime type that hints to the client that the output from our server is a JSON
def list_docs() -> list[str]:
    return list(docs.keys())


@mcp.resource(
        "docs://documents/{doc_id}", 
        mime_type="text/plain"
)
def fetch_doc(doc_id: str) -> str:
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")
    return docs[doc_id]

# Defining Prompts

- Users may ask Claude to format a document, this can be done through prompts
- The point of this, is quality of life. although users can write their own prompts format the docs, usually in MCP server, prompts are optimised to give users a better output


In [ ]:
@mcp.prompt(
    name="format",
    description="Rewrites the contents of the document in Markdown format.",
)
def format_document(
    doc_id: str = Field(description="Id of the document to format"),
) -> list[base.Message]:
    #   The optimised prompt
    prompt = f"""
    Your goal is to reformat a document to be written with markdown syntax.

    The id of the document you need to reformat is:
    <document_id>
    {doc_id}
    </document_id>

    Add in headers, bullet points, tables, etc as necessary. Feel free to add in extra text, but don't change the meaning of the report.
    Use the 'edit_document' tool to edit the document. After the document has been edited, respond with the final version of the doc. Don't explain your changes.
    """

    # return list of message as defined above in -> list[base.Messages]
    return [base.UserMessage(prompt)]

# MCP Client

Client consist of:
- MCP Client - Authoring the class to define the schema of the connection
- Client Session - Actual connection to MCP server
    - requires cleanup

# Accessing Tools

In [ ]:
async def list_tools(self) -> list[types.Tool]:
    result = await self.session().list_tools()  #self.session() gets you access to the MCP server, call a list_tools() function
    return result.tools

async def call_tool(
    self, tool_name: str, tool_input
) -> types.CallToolResult | None:
    return await self.session().call_tool(tool_name, tool_input)

# Tesing tool

In [ ]:
# For testing
async def main():
    async with MCPClient(
        # If using Python without UV, update command to 'python' and remove "run" from args.
        command="uv",
        args=["run", "mcp_server.py"],
    ) as _client:
        pass

# Accessing Resources

In [ ]:
from pydantic import AnyUrl
import json

async def read_resource(self, uri: str) -> Any:
    result = await self.session().read_resource(AnyUrl(uri))    # AnyURL is for typeError fix
    resource = result.contents[0]

    if isinstance(resource, types.TextResourceContents):
        if resource.mimeType == "application/json": # If defined resource hints at json, then load json
            return json.loads(resource.text)


# Accessing Prompts

In [ ]:
async def list_prompts(self) -> list[types.Prompt]:
        result = await self.session().list_prompts()
        return result.prompts

async def get_prompt(self, prompt_name, args: dict[str, str]):
    result = await self.session().get_prompt(prompt_name, args)
    return result.messages

# Recap

**Tools:**
- Claude (model) decides when to use tools
- used to give additional functions

**Resources:**
- App interface decides when to call resources (meaning both the model and you decide)
- Get data into app/ add context to chat

**Prompts:**
- User decides when to use
- create slash commands, button click or menu option based on user input
